Listado de tablas

In [1]:
import sqlite3
import pandas as pd 

# Conexion a la fuente de datos crudos
conn = sqlite3.connect('../data/database.sqlite') 

# Consultar las tablas disponibles
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type = 'table' ;", conn)
print("Tablas en la base de datos:")
print(tables)

# Cerramos la conexion
conn.close()

Tablas en la base de datos:
                name
0    sqlite_sequence
1  Player_Attributes
2             Player
3              Match
4             League
5            Country
6               Team
7    Team_Attributes


In [ ]:
conn = sqlite3.connect('../data/database.sqlite')

query = """
SELECT 
    m.date AS Fecha, 
    l.name AS Liga, 
    ht.team_long_name AS Equipo_Local, 
    at.team_long_name AS Equipo_Visitante,
    m.home_team_goal AS Goles_Local, 
    m.away_team_goal AS Goles_Visitante, 
    m.B365H AS Cuota_Local,
    m.B365D AS Cuota_Empate, 
    m.B365A AS Cuota_Visitante
FROM Match m 
JOIN League l ON m.league_id = l.id
JOIN Team ht ON m.home_team_api_id = ht.team_api_id
JOIN Team at ON m.away_team_api_id = at.team_api_id
WHERE l.name = 'England Premier League' 
ORDER BY m.date DESC;
"""

# Pandas ejecuta el SQL y lo convierte en un DataFrame
df_partidos = pd.read_sql_query(query, conn)

conn.close()

# Imprimimos 5 primeras filas 
print(f"Tamaño del dataset extraído: {df_partidos.shape}")
df_partidos.head()



Tamaño del dataset extraído: (3040, 9)


,Fecha,Liga,Equipo_Local,Equipo_Visitante,Goles_Local,Goles_Visitante,Cuota_Local,Cuota_Empate,Cuota_Visitante
0,2016-05-17 00:00:00,England Premier League,Manchester United,Bournemouth,3,1,1.67,4.20,5.25
1,2016-05-15 00:00:00,England Premier League,Arsenal,Aston Villa,4,0,1.17,9.00,17.00
2,2016-05-15 00:00:00,England Premier League,Chelsea,Leicester City,1,1,2.30,3.75,3.10
3,2016-05-15 00:00:00,England Premier League,Everton,Norwich City,3,0,1.80,4.00,4.50
4,2016-05-15 00:00:00,England Premier League,Newcastle United,Tottenham Hotspur,5,1,4.50,4.00,1.80


In [4]:
import numpy as np

# 1. Diagnóstico real de valores nulos en todo el dataset
print("Valores nulos por columna:")
print(df_partidos.isnull().sum())

# 2. Limpieza rápida: Si hay partidos sin cuotas, los eliminamos
df_partidos = df_partidos.dropna()

# 3. Feature Engineering: Crear la columna de 'Resultado_Real' (H=Local, D=Empate, A=Visitante)
# Esto es vital para luego comparar si nuestro modelo predijo al ganador correcto
condiciones = [
    df_partidos['Goles_Local'] > df_partidos['Goles_Visitante'],
    df_partidos['Goles_Local'] < df_partidos['Goles_Visitante']
]
opciones = ['H', 'A']
df_partidos['Resultado_Real'] = np.select(condiciones, opciones, default='D')

# 4. Ver el tamaño final después de limpiar y la nueva columna
print(f"\nTamaño del dataset limpio: {df_partidos.shape}")
df_partidos[['Equipo_Local', 'Equipo_Visitante', 'Goles_Local', 'Goles_Visitante', 'Resultado_Real']].head()

Valores nulos por columna:
Fecha               0
Liga                0
Equipo_Local        0
Equipo_Visitante    0
Goles_Local         0
Goles_Visitante     0
Cuota_Local         0
Cuota_Empate        0
Cuota_Visitante     0
dtype: int64

Tamaño del dataset limpio: (3040, 10)


,Equipo_Local,Equipo_Visitante,Goles_Local,Goles_Visitante,Resultado_Real
0,Manchester United,Bournemouth,3,1,H
1,Arsenal,Aston Villa,4,0,H
2,Chelsea,Leicester City,1,1,D
3,Everton,Norwich City,3,0,H
4,Newcastle United,Tottenham Hotspur,5,1,H
